In [1]:
import os
import re
import math
import shutil
import subprocess

from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from matplotlib.dates import DateFormatter
from dateutil.tz import tzutc, tzlocal
from scipy import stats
from scipy.optimize import brentq, curve_fit, fsolve

In [2]:
%load_ext autoreload
%autoreload 2

import official_GSSHA_Python_functions as gf

In [4]:

# Get the current working directory
cwd = os.getcwd()
script_dir = Path.cwd()  # Use  if you're in Jupyter or interactive session


GSSHA_prj_name = "Waialua_FIM_testing"
model_dir = script_dir / GSSHA_prj_name

xys_tsf_iteration_file_name = GSSHA_prj_name + "_ITERATION_file"
xys_tsf_iteration_value = "20.0"


GSSHA_executables = script_dir / 'GSSHA_applications'


### LOW FLOW ###
FLOW_TEST = [20]
low_flow_time = 1440 + 600 #tidal warmup period = 1 day = 1440 mins


gf.cleanup_model_dir(model_dir)

for flow in FLOW_TEST:
    df_prj = gf.read_prj_file(GSSHA_prj_name +".prj", prj_folder_path = model_dir)
    
    new_value = float(flow)
    cfs_flow_for_filenames = round(flow*35.31467)

    #ITERATION FILES
    gf.replace_value_in_gssha_file(read_dir=model_dir , gssha_sample_file = xys_tsf_iteration_file_name + ".tsf", save_filename = GSSHA_prj_name + "_bc.tsf", old_value =xys_tsf_iteration_value, new_value = new_value)
    gf.replace_value_in_gssha_file(read_dir= model_dir, gssha_sample_file = xys_tsf_iteration_file_name + ".xys", save_filename = GSSHA_prj_name + ".xys", old_value = xys_tsf_iteration_value, new_value = new_value)
    ##UPDATE THE ITERATION W THE NEW XYS AND TSF FILE

    #!!!!write a function to copy with specified cfs flows in name!!!

    #SAVE TEXT XYS & TSF FILES
    #These files are for the 
    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + "_bc.tsf",
        new_filename=GSSHA_prj_name + "_OUTPUT-input-flow-cfs-" + str(cfs_flow_for_filenames) + "_bc.tsf"
    )

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + "_bc.tsf",
        new_filename=GSSHA_prj_name + "_OUTPUT-input-flow-cfs-" + str(cfs_flow_for_filenames) + ".xys"
    )

    
    #SAVE OWS FILE 
    #this is the file for water surface elevation (above DEM LMSL datum) at the active USGS stream gauge
    ows = df_prj.Value['OVERLAND_WSE'].split('\\')[:-1]
    ows.append(GSSHA_prj_name + "_OUTPUT_WSE-active-USGS-gauge-m-" + str(cfs_flow_for_filenames) + ".ows")
    ows = '\\'.join(ows) + '"'

    #SAVE OTL FILE
    #this is the file for streamflow at the outlet of the watershed
    otl = df_prj.Value['OUTLET_HYDRO'].split('\\')[:-1]
    otl.append(GSSHA_prj_name + "_OUTPUT_stream-outlet-cms-" + str(cfs_flow_for_filenames) + ".otl")
    otl = '\\'.join(otl) + '"'

    #SAVE OQC FILE
    #this is the file for overland cumulative streamflow at the USGS stats location
    oqc = df_prj.Value['CUM_DISCHARGE'].split('\\')[:-1]
    oqc.append(GSSHA_prj_name + "_OUTPUT_USGS-STATS-location-cms-" + str(cfs_flow_for_filenames) + ".oqc")
    oqc = '\\'.join(oqc) + '"'

    #append the filenames/paths to the GSSHA prj file for output
    df_prj.Value['CUM_DISCHARGE'] = oqc
    df_prj.Value['OUTLET_HYDRO'] = otl
    df_prj.Value['OVERLAND_WSE'] = ows






    #OPTIMIZE FOR SIMULATION TIME : AVOID CRASHES, UNNECESSARY COMPUTATION
    
    if flow <= 50:
        flow_time = 600
    else:
        tot_time = high_flow_time
        df_prj.Value['TOT_TIME'] = tot_time
    
    df_prj = df_prj.reset_index()


    #UPDATE PRJ FILE WHICH CONTAINS THE PATHS
    gf.convert_df_to_prj(
        df_prj,
        output_folder= model_dir,
        prj_file_name=GSSHA_prj_name
    )

    gf.copy_gssha_apps_to_model(GSSHA_executables, model_dir)
    gf.run_gssha(model_dir, PROJECT_FILE = GSSHA_prj_name + ".prj")
    # move_and_rename_gssha_output(MODEL_DIR, RESULTS_DIR, output_description = "TEST_RUN" + first_date, extension = "otl")
    gf.cleanup_model_dir(model_dir)

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".dep",
        new_filename=GSSHA_prj_name + "_OUTPUT-timeseries-depth-m-" + str(cfs_flow_for_filenames) + ".dep"
    )

    gf.copy_text_file(
        folder_path=model_dir,
        original_filename=GSSHA_prj_name + ".gfl",
        new_filename=GSSHA_prj_name + "_OUTPUT-maxflood-dep-m-" + str(cfs_flow_for_filenames) + ".gfl"
    )





✔️ Cleaned up model folder.


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\bgorberg\\Documents\\GitHub\\Flood_Stage_Maps\\RUN_GSSHA\\Waialua_FIM_testing\\Waialua_FIM_testing_ITERATION_file.tsf'

In [ ]:
#READ OTL FILE :

#EVALUATE PEAK